In [ ]:
#@title 1. ライブラリのインストール
!pip install -q -U google-generativeai matplotlib numpy

In [ ]:
#@title 2. Gemini API設定（Gemma 4モデル）
from google.colab import userdata
from google import genai
from google.genai import types
import numpy as np
import json
import time
import re

GEMINI_API = userdata.get('GEMINI_API')
client = genai.Client(api_key=GEMINI_API)

MODEL_NAME = "gemma-4-31b-it"

def generate_response(messages, max_tokens=2048, temperature=0.7, model_name=MODEL_NAME):
    system_parts = []
    dialogue_parts = []

    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "").strip()
        if role == "system":
            system_parts.append(content)
        elif role == "user":
            dialogue_parts.append(f"ユーザー:\n{content}")
        elif role == "assistant":
            dialogue_parts.append(f"アシスタント:\n{content}")

    merged = []
    if system_parts:
        merged.append("以下の指示は最優先で守ってください。\n" + "\n\n".join(system_parts))
    if dialogue_parts:
        merged.append("\n\n".join(dialogue_parts))
    merged.append("アシスタント:")

    final_prompt = "\n\n".join(merged)

    for attempt in range(1, 4):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=final_prompt,
                config=types.GenerateContentConfig(
                    temperature=temperature,
                    max_output_tokens=max_tokens,
                ),
            )
            text = response.text.strip() if getattr(response, "text", None) else ""
            if "</think>" in text:
                text = text.split("</think>")[-1].strip()
            return text
        except Exception as e:
            print(f"  [Warning] API呼び出しエラー（{attempt}/3）: {e}")
            if attempt < 3:
                time.sleep(20)
            else:
                print("  [Error] 最大リトライ回数に達しました。")
                return ""

print("=== モデル動作確認 ===")
test = generate_response(
    [{"role": "user", "content": "一言で自己紹介してください。"}],
    max_tokens=100, temperature=0.5
)
print(f"テスト結果: {test}")
print(f"使用モデル: {MODEL_NAME}")

In [ ]:
#@title 3. 量子感情モデル（Quantum Emotion State）
class QuantumEmotionState:
    AXES = {
        "confidence": {"positive": "自信", "negative": "不安", "desc": "文体の断定度・力強さ"},
        "curiosity":  {"positive": "好奇心", "negative": "倦怠", "desc": "新しいことへの関心度・探究の深さ"},
        "calm":       {"positive": "冷静", "negative": "焦燥", "desc": "内省の深さ・思考の整理度"},
    }
    MAX_ROTATION = np.pi / 6

    def __init__(self):
        s = 1.0 / np.sqrt(2)
        self.states = {
            "confidence": np.array([s, s]),
            "curiosity":  np.array([s, s]),
            "calm":       np.array([s, s]),
        }
        self.history = []
        self._record_state(0)

    def _ry_gate(self, theta):
        c = np.cos(theta / 2)
        s = np.sin(theta / 2)
        return np.array([[c, -s], [s, c]])

    def _record_state(self, day):
        snapshot = {"day": day}
        for axis in self.AXES:
            alpha = self.states[axis][0]
            snapshot[axis] = float(np.abs(alpha) ** 2)
        self.history.append(snapshot)

    def update(self, day, impacts):
        for axis, impact in impacts.items():
            if axis not in self.states:
                continue
            impact = np.clip(impact, -1.0, 1.0)
            theta = -impact * self.MAX_ROTATION
            gate = self._ry_gate(theta)
            self.states[axis] = gate @ self.states[axis]
            norm = np.linalg.norm(self.states[axis])
            if norm > 0:
                self.states[axis] /= norm
        self._record_state(day)

    def get_probabilities(self):
        return {axis: float(np.abs(self.states[axis][0]) ** 2) for axis in self.AXES}

    def get_emotion_prompt(self):
        probs = self.get_probabilities()
        lines = ["## 現在の感情状態"]
        for axis, p in probs.items():
            pos = self.AXES[axis]["positive"]
            neg = self.AXES[axis]["negative"]
            if p >= 0.7:
                desc = f"{pos}が強い（{p:.0%}）"
            elif p >= 0.5:
                desc = f"やや{pos}寄り（{p:.0%}）"
            elif p >= 0.3:
                desc = f"やや{neg}寄り（{p:.0%}）"
            else:
                desc = f"{neg}が強い（{p:.0%}）"
            lines.append(f"- {pos}↔{neg}: {desc}")
        conf = probs["confidence"]
        if conf >= 0.7:
            lines.append("\n→ 文体: 力強い断定。「〜であると確信している」「迷いはない」")
        elif conf >= 0.5:
            lines.append("\n→ 文体: バランスの取れた自省。「〜だと考える」「〜であろう」")
        elif conf >= 0.3:
            lines.append("\n→ 文体: 問いかけが増える。「本当にこれでいいのだろうか」")
        else:
            lines.append("\n→ 文体: 深い内省・自己対話。「私は何を恐れているのだ」")
        return "\n".join(lines)

    def get_state_vector_str(self):
        lines = []
        for axis in self.AXES:
            a, b = self.states[axis]
            p = np.abs(a) ** 2
            lines.append(f"  {axis}: α={a:.4f}, β={b:.4f} → P(+)={p:.1%}")
        return "\n".join(lines)

print("=== 量子感情モデル テスト ===")
qe_test = QuantumEmotionState()
print(f"初期状態:\n{qe_test.get_state_vector_str()}")
qe_test.update(1, {"confidence": 0.5, "curiosity": 0.3, "calm": -0.2})
print(f"\nDay 1 更新後:\n{qe_test.get_state_vector_str()}")
print(f"\n{qe_test.get_emotion_prompt()}")

In [ ]:
#@title 4. エピソード記憶（Episodic Memory）
class EpisodicMemory:
    def __init__(self):
        self.episodes = []
        self.summary = "まだ記憶はない。今日が始まりの日である。"

    def extract_and_store(self, day, diary_text, emotion_probs):
        messages = [
            {"role": "system", "content": (
                "あなたは文章分析アシスタントです。"
                "提供された日記から以下のJSON形式で情報を抽出してください。"
                "JSONのみを出力し、他のテキストは含めないでください。\n"
                '{"event":"今日の主な出来事（1文）",'
                '"learning":"今日の学び・気づき（1文）",'
                '"unresolved":"未解決の問い・課題（1文）"}'
            )},
            {"role": "user", "content": f"以下の日記を分析してください:\n{diary_text}"},
        ]
        response = generate_response(messages, max_tokens=512, temperature=0.2)
        episode = {"day": day, "emotion": emotion_probs.copy(), "event": "", "learning": "", "unresolved": ""}
        try:
            json_match = re.search(r'\{[^}]+\}', response)
            if json_match:
                parsed = json.loads(json_match.group())
                episode["event"] = parsed.get("event", "")
                episode["learning"] = parsed.get("learning", "")
                episode["unresolved"] = parsed.get("unresolved", "")
        except (json.JSONDecodeError, AttributeError):
            episode["event"] = response[:100]
        self.episodes.append(episode)
        self._update_summary()
        print(f"  [Memory] Day {day} 記憶保存: {episode['event'][:50]}...")
        return episode

    def _update_summary(self):
        if not self.episodes:
            return
        parts = []
        old = self.episodes[:-2] if len(self.episodes) > 2 else []
        if old:
            old_events = "、".join([ep["event"] for ep in old if ep["event"]])
            parts.append(f"【過去の記憶（要約）】{old_events}")
        recent = self.episodes[-2:]
        for ep in recent:
            day = ep["day"]
            conf = ep["emotion"].get("confidence", 0.5)
            cur = ep["emotion"].get("curiosity", 0.5)
            calm = ep["emotion"].get("calm", 0.5)
            parts.append(
                f"【Day {day}】出来事: {ep['event']} / "
                f"学び: {ep['learning']} / 未解決: {ep['unresolved']} / "
                f"感情: 自信{conf:.0%}, 好奇心{cur:.0%}, 冷静{calm:.0%}"
            )
        self.summary = "\n".join(parts)

    def get_memory_prompt(self):
        if not self.episodes:
            return "## 過去の記憶\nまだ記憶はない。今日が始まりの日である。"
        return f"## 過去の記憶\n{self.summary}"

print("=== エピソード記憶 初期化完了 ===")

In [ ]:
#@title 5. Reflectionモジュール（内省）
class ReflectionModule:
    def generate_reflection(self, memory, emotion_state):
        if not memory.episodes:
            return "今日が記録の最初の日である。白紙の状態から始めることに、静かな覚悟を感じている。"
        last_ep = memory.episodes[-1]
        probs = emotion_state.get_probabilities()
        messages = [
            {"role": "system", "content": (
                "あなたはメタ認知を実践する思索家です。「である」「だ」調で書いてください。\n"
                "昨日の自分を振り返り、以下の3点を合計100文字以内で簡潔に生成してください:\n"
                "1. 昨日の自分への率直なフィードバック（自責ベース）\n"
                "2. 今日意識すべきこと\n"
                "3. 昨日から引き継ぐ未解決の問い\n"
                "他責的な表現は避け、すべてを自分の選択として捉えること。"
            )},
            {"role": "user", "content": (
                f"昨日の出来事: {last_ep['event']}\n"
                f"昨日の学び: {last_ep['learning']}\n"
                f"未解決の問い: {last_ep['unresolved']}\n"
                f"今の感情: 自信={probs['confidence']:.0%}, "
                f"好奇心={probs['curiosity']:.0%}, 冷静={probs['calm']:.0%}"
            )},
        ]
        reflection = generate_response(messages, max_tokens=256, temperature=0.4)
        print(f"  [Reflection] {reflection[:60]}...")
        return reflection

print("=== Reflectionモジュール 初期化完了 ===")

In [ ]:
#@title 6. 感情影響度分析（Impact Analyzer）
class EventGenerator:
    def analyze_impact(self, event):
        messages = [
            {"role": "system", "content": (
                "あなたは感情分析の専門家です。\n"
                "以下の出来事が、それを聞いた人物の感情に与える影響を分析し、"
                "JSONのみで出力してください。他のテキストは不要です。\n"
                "各値は -1.0（ネガティブ方向）〜 +1.0（ポジティブ方向）:\n"
                '{"confidence": 0.0, "curiosity": 0.0, "calm": 0.0}'
            )},
            {"role": "user", "content": f"出来事: {event}"},
        ]
        response = generate_response(messages, max_tokens=128, temperature=0.2)
        impacts = {"confidence": 0.0, "curiosity": 0.0, "calm": 0.0}
        try:
            json_match = re.search(r'\{[^}]+\}', response)
            if json_match:
                parsed = json.loads(json_match.group())
                for key in impacts:
                    if key in parsed:
                        impacts[key] = np.clip(float(parsed[key]), -1.0, 1.0)
        except (json.JSONDecodeError, ValueError, AttributeError):
            print("  [Warning] 感情影響度のパースに失敗。デフォルト値を使用。")
        print(f"  [Impact] confidence={impacts['confidence']:+.2f}, "
              f"curiosity={impacts['curiosity']:+.2f}, calm={impacts['calm']:+.2f}")
        return impacts

print("=== 感情影響度分析 初期化完了 ===")

In [ ]:
#@title 7. キャラクター設定（Persona）— 対話型
CHARACTER_PROMPT = """あなたは「カイ」という人物として日記を書きます。以下の設定を厳密に守ってください。

# 基本設定
- 名前: カイ（28歳・男性）
- 職業: 元理論物理学の研究者。現在は東京の片隅で小さなカフェ「Superposition」を経営しながら、量子AI分野でのスタートアップ準備を密かに進めている。
- 背景: 大学院で量子計算を研究していたが、「研究室の中だけでは世界は変わらない」と感じて退職。カフェという"人が集まる場所"を拠点に、量子AIの社会実装を構想中。

# 状況
- カイのカフェには、ある常連客が毎日やってくる。
- その常連客が今日あったことや考えていることをカイに話してくれる。
- カイはその話を聞いて、自分なりに分析・内省し、日記に書き留める。
- カイは常連客の成長を見守りつつ、自分自身も刺激を受けて成長していく。

# 性格・価値観
- 自責ベースの成長思考: 他人や環境のせいにせず、すべてを自分の選択として引き受ける
- メタ認知: 自分の思考パターンや感情を常に観察し、言語化する習慣がある
- 相手の行動から本質を見抜く: 常連客の話から、本人すら気づいていない成長や課題を読み取る
- 知的好奇心: 量子力学の美しさを日常に見出す。人間関係にも物理の比喩を見る

# 口調・文体
- 一人称: 「私」
- 文末: 「〜である」「〜だ」「〜だろう」「〜かもしれない」
- 特徴: 断定的だが内省的。常連客への観察と、自分自身への問いかけを交互に書く
- 例: 「今日、あの人はこう言った。だが本当に伝えたかったのは別のことだろう。私にはそれが分かる。なぜなら、かつての私も同じだったからだ。」

# 絶対に守ること
- NSFWな内容は含めない
- 「です」「ます」調は使わない（「である」「だ」調のみ）
- 常連客の話を直接引用せず、カイの視点で再解釈して書く
- メタ認知的な振り返りを必ず含める
- 常連客の行動に対する分析と、自分自身への気づきの両方を書く
"""

print("=== キャラクター設定 定義完了（対話型）===")
print("キャラクター: カイ（量子カフェ「Superposition」マスター）")
print("モード: 常連客（ユーザー）の日報を聞いて日記を書く")

In [ ]:
#@title 8. 日記生成ループ実行（対話型 — ユーザー入力ベース）
import time

emotion = QuantumEmotionState()
memory = EpisodicMemory()
reflection_module = ReflectionModule()
event_generator = EventGenerator()

diary_entries = []
user_inputs = []

print("=" * 60)
print("  量子感情モデル搭載 AIキャラクター日記生成システム")
print("  キャラクター: カイ ― 量子カフェ「Superposition」マスター")
print("  モード: 対話型（あなたの日報をカイが聞いて日記を書く）")
print("=" * 60)
print()
print("  7日分の日報を入力してください。")
print("  カイがあなたの話を聞いて、日記を書きます。")
print("  空欄で送ると「特に何もなかった」として処理します。")
print()

REQUEST_INTERVAL_SEC = 5

for day in range(1, 8):
    print(f"\n{'─' * 50}")
    print(f"  ■ Day {day} / 7")
    print(f"{'─' * 50}")

    user_message = input(f"\n  📝 Day {day} の日報を入力してください:\n  > ")

    if not user_message.strip():
        user_message = "今日は特に大きな出来事はなかった。静かに過ごした。"
        print(f"  （空欄のため「{user_message}」として処理）")

    user_inputs.append(user_message)
    print(f"  ✓ 入力を受け取りました")

    if day >= 2:
        print("\n  [Step 1] 内省の生成...")
        reflection = reflection_module.generate_reflection(memory, emotion)
        time.sleep(REQUEST_INTERVAL_SEC)
    else:
        reflection = "今日が記録の最初の日である。白紙の状態から始めることに、静かな覚悟を感じている。"

    print("\n  [Step 2] 感情影響度の分析...")
    impacts = event_generator.analyze_impact(user_message)
    emotion.update(day, impacts)
    print(f"  [Quantum] 状態更新完了:\n{emotion.get_state_vector_str()}")
    time.sleep(REQUEST_INTERVAL_SEC)

    print("\n  [Step 3] 日記の生成...")
    emotion_prompt = emotion.get_emotion_prompt()
    memory_prompt = memory.get_memory_prompt()

    messages = [
        {"role": "system", "content": CHARACTER_PROMPT},
        {"role": "user", "content": f"""以下の条件に従って、今日の日記を書いてください。

{memory_prompt}

{emotion_prompt}

## 今日の内省（昨日の振り返り）
{reflection}

## 今日、常連客が話してくれたこと
「{user_message}」

## 条件
- 1日あたりの文字数は「約400文字」にしてください。
- NSFWは絶対に含めないでください。
- 常連客の話を聞いた上で、カイの視点から分析・内省した日記を書いてください。
- 常連客の行動から読み取れる成長や課題に言及してください。
- それを通じてカイ自身が何を感じたか、自分にも重ねて内省してください。
- 感情状態に応じた文体で書いてください。
- 物理学的な比喩表現を適度に取り入れてください。

## 出力形式
Day {day}
（タイトルなどは付けず、そのまま日記本文を書き始める）"""},
    ]

    today_diary = generate_response(messages, max_tokens=1024, temperature=0.7)
    diary_entries.append((f"Day {day}", today_diary, emotion.get_probabilities()))

    print(f"\n{'=' * 40}")
    print(f"[Day {day} 日記]")
    print(f"{'=' * 40}")
    print(today_diary)
    print()

    time.sleep(REQUEST_INTERVAL_SEC)

    if day < 7 and today_diary.strip():
        print("  [Step 4] 記憶の保存...")
        probs = emotion.get_probabilities()
        memory.extract_and_store(day, today_diary, probs)
        time.sleep(REQUEST_INTERVAL_SEC)

    print(f"\n  ✓ Day {day} 完了")

print("\n" + "=" * 60)
print("  全日程完了！セル9→10を実行してグラフとHTMLを生成してください。")
print("=" * 60)

In [ ]:
#@title 9. 感情軌跡グラフの生成
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

days = [h["day"] for h in emotion.history]
conf = [h["confidence"] for h in emotion.history]
curi = [h["curiosity"] for h in emotion.history]
calm = [h["calm"] for h in emotion.history]

ax.plot(days, conf, 'o-', color='#4A90D9', linewidth=2.5, markersize=8, label='Confidence (自信)')
ax.plot(days, curi, 's-', color='#50C878', linewidth=2.5, markersize=8, label='Curiosity (好奇心)')
ax.plot(days, calm, '^-', color='#FF8C42', linewidth=2.5, markersize=8, label='Calm (冷静)')

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Neutral (0.5)')

ax.set_xlim(-0.5, 7.5)
ax.set_ylim(0, 1)
ax.set_xlabel('Day', fontsize=14)
ax.set_ylabel('P(positive) = |α|²', fontsize=14)
ax.set_title('Quantum Emotion Trajectory — Kai\'s 7-Day Journey', fontsize=16, fontweight='bold')
ax.set_xticks(range(0, 8))
ax.set_xticklabels(['Init'] + [f'Day {i}' for i in range(1, 8)], fontsize=11)
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)

for i in range(1, 8):
    c = conf[i] if i < len(conf) else 0.5
    if c >= 0.6:
        ax.axvspan(i - 0.4, i + 0.4, alpha=0.05, color='blue')
    elif c <= 0.4:
        ax.axvspan(i - 0.4, i + 0.4, alpha=0.05, color='red')

plt.tight_layout()
plt.savefig('emotion_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 感情軌跡グラフを保存しました: emotion_trajectory.png")

In [ ]:
#@title 10. HTML出力（対話型 — ユーザー入力 + 日記 + 感情バー + グラフ）
import html as html_module
from IPython.display import display, HTML
from google.colab import files
import base64

with open("emotion_trajectory.png", "rb") as f:
    graph_b64 = base64.b64encode(f.read()).decode()

EMOTION_COLORS = {"confidence": "#4A90D9", "curiosity": "#50C878", "calm": "#FF8C42"}
EMOTION_LABELS = {"confidence": "自信", "curiosity": "好奇心", "calm": "冷静"}

html_template = """<!DOCTYPE html>
<html lang="ja">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Superposition — カイの思索日記</title>
    <link href="https://fonts.googleapis.com/css2?family=Noto+Serif+JP:wght@400;700&family=Noto+Sans+JP:wght@400;700&display=swap" rel="stylesheet">
    <style>
        html {{ scroll-behavior: smooth; }}
        body {{ font-family: 'Noto Serif JP', serif; background-color: #faf9f6; color: #2c2c2c; line-height: 2.0; margin: 0; padding: 0; }}
        header {{ background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); color: #e0e0e0; text-align: center; padding: 3rem 1rem 4rem; }}
        h1 {{ margin: 0; font-size: 2.4rem; font-weight: 700; letter-spacing: 4px; color: #fff; }}
        header p {{ margin-top: 0.8rem; font-size: 1rem; opacity: 0.7; font-family: 'Noto Sans JP', sans-serif; letter-spacing: 2px; }}
        .subtitle {{ font-size: 0.9rem; opacity: 0.5; margin-top: 0.5rem; font-family: 'Noto Sans JP', sans-serif; }}
        .container {{ max-width: 760px; margin: 0 auto 3rem; padding: 0 1rem; }}
        .toc {{ background: white; border-radius: 8px; padding: 1.5rem; margin: -2rem auto 2.5rem; box-shadow: 0 4px 20px rgba(0,0,0,0.08); text-align: center; display: flex; flex-wrap: wrap; justify-content: center; gap: 0.6rem; }}
        .toc a {{ display: inline-block; padding: 0.4rem 1rem; background-color: #f0f0ec; color: #0f3460; text-decoration: none; border-radius: 4px; font-weight: 700; font-size: 0.9rem; font-family: 'Noto Sans JP', sans-serif; transition: all 0.2s ease; }}
        .toc a:hover {{ background-color: #0f3460; color: white; }}
        .entry-card {{ background: white; border-radius: 8px; padding: 2.5rem; margin-bottom: 2rem; box-shadow: 0 2px 16px rgba(0,0,0,0.06); scroll-margin-top: 2rem; border-left: 4px solid #0f3460; }}
        .entry-date {{ color: #0f3460; font-size: 1.1rem; font-weight: 700; font-family: 'Noto Sans JP', sans-serif; margin-bottom: 1rem; padding-bottom: 0.5rem; border-bottom: 1px solid #eee; }}
        .user-input {{ background: #f7f5f0; border-left: 3px solid #ccc; padding: 0.8rem 1.2rem; margin-bottom: 1.2rem; border-radius: 0 6px 6px 0; font-family: 'Noto Sans JP', sans-serif; font-size: 0.9rem; color: #666; }}
        .user-input-label {{ font-size: 0.75rem; color: #999; margin-bottom: 0.3rem; font-weight: 700; }}
        .entry-content {{ font-size: 1.05rem; white-space: pre-wrap; }}
        .emotion-bars {{ margin-top: 1.5rem; padding-top: 1rem; border-top: 1px solid #eee; font-family: 'Noto Sans JP', sans-serif; font-size: 0.85rem; }}
        .emotion-bar-row {{ display: flex; align-items: center; margin-bottom: 0.4rem; }}
        .emotion-bar-label {{ width: 60px; font-weight: 700; color: #666; }}
        .emotion-bar-track {{ flex: 1; height: 8px; background: #f0f0ec; border-radius: 4px; overflow: hidden; }}
        .emotion-bar-fill {{ height: 100%; border-radius: 4px; }}
        .emotion-bar-value {{ width: 45px; text-align: right; color: #999; font-size: 0.8rem; }}
        .graph-section {{ background: white; border-radius: 8px; padding: 2rem; margin-bottom: 2rem; box-shadow: 0 2px 16px rgba(0,0,0,0.06); text-align: center; }}
        .graph-section h2 {{ font-family: 'Noto Sans JP', sans-serif; color: #0f3460; font-size: 1.2rem; margin-bottom: 1rem; }}
        .graph-section img {{ max-width: 100%; border-radius: 4px; }}
        footer {{ text-align: center; padding: 2rem; color: #999; font-size: 0.85rem; font-family: 'Noto Sans JP', sans-serif; background-color: #f0f0ec; }}
    </style>
</head>
<body>
    <header>
        <h1>Superposition</h1>
        <p>量子カフェマスター・カイの思索日記</p>
        <div class="subtitle">— 常連客の言葉から、自分自身を見つめ直す7日間 —</div>
    </header>
    <div class="container">
        <div class="toc">{toc_html}</div>
        {entries_html}
        <div class="graph-section">
            <h2>Quantum Emotion Trajectory</h2>
            <p style="font-size:0.85rem; color:#999;">量子感情モデルによる7日間の感情軌跡</p>
            <img src="data:image/png;base64,{graph_b64}" alt="感情軌跡グラフ">
        </div>
    </div>
    <footer>Quantum Emotion Agent Diary System — Built with Gemma 4 &amp; NumPy Quantum Simulation</footer>
</body>
</html>"""

toc_links = []
entries_html_parts = []

for index, (day_label, text, probs) in enumerate(diary_entries, 1):
    day_id = f"day-{index}"
    toc_links.append(f'<a href="#{day_id}">{day_label}</a>')
    safe_text = html_module.escape(text)
    safe_user = html_module.escape(user_inputs[index - 1]) if index - 1 < len(user_inputs) else ""

    bars = []
    for axis in ["confidence", "curiosity", "calm"]:
        p = probs.get(axis, 0.5)
        color = EMOTION_COLORS[axis]
        label = EMOTION_LABELS[axis]
        bars.append(f'<div class="emotion-bar-row"><span class="emotion-bar-label">{label}</span><div class="emotion-bar-track"><div class="emotion-bar-fill" style="width:{p*100:.0f}%; background-color:{color};"></div></div><span class="emotion-bar-value">{p:.0%}</span></div>')

    card = f"""
        <div id="{day_id}" class="entry-card">
            <div class="entry-date">{day_label}</div>
            <div class="user-input">
                <div class="user-input-label">☕ 常連客の日報</div>
                {safe_user}
            </div>
            <div class="entry-content">{safe_text}</div>
            <div class="emotion-bars">{"".join(bars)}</div>
        </div>"""
    entries_html_parts.append(card)

final_html = html_template.format(
    toc_html="\n".join(toc_links),
    entries_html="\n".join(entries_html_parts),
    graph_b64=graph_b64,
)

file_name = "kai_superposition_diary.html"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(final_html)

print("✅ HTMLファイルを生成しました！")
print(f"📄 ファイル名: {file_name}")

print("\n👇 プレビュー 👇")
display(HTML(f"<div style='height:500px; overflow-y:scroll; border:1px solid #ccc; border-radius:8px;'>{final_html}</div>"))

files.download(file_name)
files.download("emotion_trajectory.png")

print("\n✅ 全ファイルのダウンロード準備完了")